# Transformación unificada EAL

Notebook común para transformar los CSV de formación de la Encuesta Anual Laboral que ya están extraídos en `Equip_31/Data/pre_processed_2020_2024/EAL_15062026/{anio}`.

Este notebook no descarga datos, no lee los Excel originales y no crea archivos nuevos por defecto. Solo toma los CSV intermedios y aplica la función de transformación que corresponde a cada hoja.

La idea es unificar el trabajo hecho en los notebooks de Fede y Sabina para poder repetir el proceso por año sin duplicar código.


## Librerías y configuración

In [1]:
from pathlib import Path
import re
import pandas as pd


def encontrar_raiz_repo(inicio=None):
    ruta = Path(inicio or Path.cwd()).resolve()
    for candidata in [ruta, *ruta.parents]:
        if (candidata / "Equip_31").exists():
            return candidata
    raise FileNotFoundError("No se encontró la raíz del repositorio con Equip_31.")


REPO_ROOT = encontrar_raiz_repo()
PRE_PROCESSED_BASE = REPO_ROOT / "Equip_31/Data/pre_processed_2020_2024/EAL_15062026"
PROCESSED_BASE = REPO_ROOT / "Equip_31/Data/Processed_2020_2024/EAL_15062026"

# Si se quiere trabajar un solo año, por ejemplo 2023, cambiar por: ANIOS = [2023]
def detectar_anios_pre_processed():
    if not PRE_PROCESSED_BASE.exists():
        return []
    return sorted(
        int(carpeta.name)
        for carpeta in PRE_PROCESSED_BASE.iterdir()
        if carpeta.is_dir() and carpeta.name.isdigit()
    )


ANIOS = [2023] #Cambiamos por el anio q queremos trabajar

HOJAS_FORMACION = [
    "EAL-16", "EAL-17", "EAL-18", "EAL-18a", "EAL-18b", "EAL-18c",
    "EAL-19", "EAL-20", "EAL-21", "EAL-22", "EAL-23", "EAL-24", "EAL-25",
]

GUARDAR_OUTPUTS = False

print("Raíz repo:", REPO_ROOT)
print("Base pre_processed:", PRE_PROCESSED_BASE.relative_to(REPO_ROOT))
print("Años detectados en pre_processed:", ANIOS)


Raíz repo: /Users/fedeur/ProjecteData
Base pre_processed: Equip_31/Data/pre_processed_2020_2024/EAL_15062026
Años detectados en pre_processed: [2023]


## Funciones comunes

Funciones compartidas para limpiar texto, leer CSV desde `pre_processed` y pasar tablas a formato largo.


In [2]:
def limpiar_texto(valor):
    if pd.isna(valor):
        return ""
    valor = str(valor).replace("\n", " ")
    valor = re.sub(r"\s+", " ", valor)
    return valor.strip()


def normalizar_columna(valor):
    valor = limpiar_texto(valor).upper()
    valor = re.sub(r"\s+", " ", valor)
    return valor


def existe_csv_pre_processed(anio, hoja):
    return (PRE_PROCESSED_BASE / str(anio) / f"{hoja}.csv").exists()


def leer_csv_eal(anio, hoja):
    ruta_csv = PRE_PROCESSED_BASE / str(anio) / f"{hoja}.csv"
    if not ruta_csv.exists():
        raise FileNotFoundError(f"No existe el CSV esperado en pre_processed: {ruta_csv}")

    df = pd.read_csv(ruta_csv, header=None, dtype=str, keep_default_na=False, encoding="utf-8")
    df = df.map(limpiar_texto)
    df = df.replace("", pd.NA)
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")
    return df.fillna("").reset_index(drop=True)


def extraer_tabla_pregunta(texto):
    texto = limpiar_texto(texto)
    match = re.match(r"^(EAL-\d+[A-Za-z]?)\.\s*(.*)$", texto)
    if not match:
        return "", texto
    return match.group(1), match.group(2).strip()


def detectar_titulo(df, hoja):
    patron = rf"^{re.escape(hoja)}\."
    for valor in df.stack().dropna().astype(str).tolist():
        texto = limpiar_texto(valor)
        if re.match(patron, texto, flags=re.IGNORECASE):
            return texto
    return ""


def metadatos_base(anio, tabla, pregunta, ambito):
    return {
        "anio": anio,
        "tabla": tabla,
        "pregunta": pregunta,
        "ambito": ambito,
    }


def hacer_columnas_unicas(columnas):
    resultado = []
    contador = {}
    for columna in columnas:
        columna = normalizar_columna(columna) or "SIN_TITULO"
        if columna not in contador:
            contador[columna] = 1
            resultado.append(columna)
        else:
            contador[columna] += 1
            resultado.append(f"{columna}_{contador[columna]}")
    return resultado


def convertir_numerico(serie):
    return pd.to_numeric(
        serie.astype(str).str.replace(",", ".", regex=False).str.replace("%", "", regex=False),
        errors="coerce",
    )


def to_long_format(df, id_vars, value_vars, var_name="variable", value_name="porcentaje"):
    df_long = df.melt(
        id_vars=id_vars,
        value_vars=value_vars,
        var_name=var_name,
        value_name=value_name,
    )
    df_long[value_name] = convertir_numerico(df_long[value_name])
    return df_long.dropna(subset=[value_name]).reset_index(drop=True)


## Funciones de transformación de grado: EAL-16, EAL-23, EAL-24 y EAL-25

Hojas con bloques repetidos y columnas `TOTAL`, `NADA`, `POCO`, `BASTANTE`, `MUCHO`.


In [3]:
COLUMNAS_GRADO = ["TOTAL", "NADA", "POCO", "BASTANTE", "MUCHO"]


def fila_es_cabecera_grado(row):
    valores = [normalizar_columna(x) for x in row.tolist()]
    return set(COLUMNAS_GRADO).issubset(set(valores))


def process_grado_wide(anio, hoja):
    df = leer_csv_eal(anio, hoja)
    registros = []
    tabla_actual = None
    pregunta_actual = None
    columnas_actuales = None

    for _, row in df.iterrows():
        primera_celda = limpiar_texto(row.iloc[0])
        tabla_detectada, pregunta_detectada = extraer_tabla_pregunta(primera_celda)

        if re.match(rf"^{re.escape(hoja)}[A-Za-z]?$", tabla_detectada, flags=re.IGNORECASE):
            tabla_actual = tabla_detectada
            pregunta_actual = pregunta_detectada
            columnas_actuales = None
            continue

        if tabla_actual and fila_es_cabecera_grado(row):
            columnas_actuales = [normalizar_columna(x) for x in row.tolist()]
            continue

        if not (tabla_actual and columnas_actuales):
            continue

        ambito = primera_celda
        if ambito == "" or ambito.startswith("("):
            continue

        registro = metadatos_base(anio, tabla_actual, pregunta_actual, ambito)
        for posicion, columna in enumerate(columnas_actuales):
            if columna in COLUMNAS_GRADO:
                registro[columna] = pd.to_numeric(limpiar_texto(row.iloc[posicion]), errors="coerce")
        registros.append(registro)

    columnas = ["anio", "tabla", "pregunta", "ambito", *COLUMNAS_GRADO]
    if not registros:
        return pd.DataFrame(columns=columnas)
    return pd.DataFrame(registros)[columnas]



def process_eal16_auto(anio, hoja):
    df_grado = process_grado_wide(anio, hoja)
    if df_grado.shape[0] > 0:
        return df_grado

    # En 2020 EAL-16 tiene otro formato: competencias importantes por tamaño de empresa.
    df = leer_csv_eal(anio, hoja)
    mask_header = df.apply(
        lambda row: row.astype(str).str.contains("TOTAL|TRABAJADORES", case=False, regex=True).sum() >= 2,
        axis=1,
    )
    if not mask_header.any():
        return df_grado

    fila_cabecera = df.index[mask_header][0]
    return process_hoja_ancha_simple(anio, hoja, fila_cabecera=fila_cabecera, fila_inicio_datos=fila_cabecera + 1)


## Funciones de transformación de categorías: EAL-17 y EAL-18

Hojas con cabeceras agrupadas y categorías por total, tamaño, sector y CCAA.


In [4]:
SECCIONES_FILA = {
    "TAMAÑO DE LA EMPRESA": "tamano_empresa",
    "ACTIVIDAD ECONÓMICA": "sector",
    "COMUNIDAD AUTÓNOMA": "ccaa",
}


def nombre_ambito(categoria, seccion_actual):
    if categoria.upper() == "TOTAL":
        return "total_empresas"
    return seccion_actual or "total_empresas"


def columnas_agrupadas(df, fila_grupo, fila_subgrupo, primera_columna_valor=1):
    grupo_superior = [limpiar_texto(x) for x in df.iloc[fila_grupo].tolist()]
    subvariable = [limpiar_texto(x) for x in df.iloc[fila_subgrupo].tolist()]

    columnas = {}
    grupo_actual = ""
    for col in range(primera_columna_valor, df.shape[1]):
        if grupo_superior[col]:
            grupo_actual = grupo_superior[col]
        sub = subvariable[col]

        if col == primera_columna_valor and not sub:
            nombre = "TOTAL"
        elif grupo_actual and sub:
            nombre = f"{grupo_actual} - {sub}"
        else:
            nombre = grupo_actual or sub or "TOTAL"

        columnas[col] = normalizar_columna(nombre)
    return columnas


def process_eal17_18_wide(anio, hoja):
    df = leer_csv_eal(anio, hoja)
    titulo = detectar_titulo(df, hoja)
    tabla, pregunta = extraer_tabla_pregunta(titulo)
    columnas_valor = columnas_agrupadas(df, fila_grupo=4, fila_subgrupo=5)

    registros = []
    seccion_actual = None
    for _, row in df.iloc[6:].iterrows():
        categoria = limpiar_texto(row.iloc[0])
        if categoria == "" or categoria.startswith("("):
            continue

        categoria_norm = categoria.upper()
        if categoria_norm in SECCIONES_FILA:
            seccion_actual = SECCIONES_FILA[categoria_norm]
            continue

        registro = metadatos_base(anio, tabla, pregunta, nombre_ambito(categoria, seccion_actual))
        registro["categoria"] = categoria

        for col, nombre_columna in columnas_valor.items():
            valor = limpiar_texto(row.iloc[col]) if col < len(row) else ""
            registro[nombre_columna] = pd.to_numeric(valor, errors="coerce")
        registros.append(registro)

    columnas_base = ["anio", "tabla", "pregunta", "ambito", "categoria"]
    columnas_valores = [nombre for _, nombre in sorted(columnas_valor.items())]
    return pd.DataFrame(registros)[columnas_base + columnas_valores]


## Función de transformación ancho simple: EAL-18a, EAL-18b, EAL-18c, EAL-19 y EAL-20


In [5]:
def process_hoja_ancha_simple(anio, hoja, fila_cabecera, fila_inicio_datos):
    df = leer_csv_eal(anio, hoja)
    titulo = detectar_titulo(df, hoja)
    tabla, pregunta = extraer_tabla_pregunta(titulo)
    columnas_valor = hacer_columnas_unicas(df.iloc[fila_cabecera, 1:].tolist())

    registros = []
    seccion_actual = None
    for _, row in df.iloc[fila_inicio_datos:].iterrows():
        categoria = limpiar_texto(row.iloc[0])
        if categoria == "" or categoria.startswith("("):
            continue

        categoria_norm = categoria.upper()
        if categoria_norm in SECCIONES_FILA:
            seccion_actual = SECCIONES_FILA[categoria_norm]
            continue

        registro = metadatos_base(anio, tabla, pregunta, nombre_ambito(categoria, seccion_actual))
        registro["categoria"] = categoria

        for offset, columna in enumerate(columnas_valor, start=1):
            valor = limpiar_texto(row.iloc[offset]) if offset < len(row) else ""
            registro[columna] = pd.to_numeric(valor, errors="coerce")

        if all(pd.isna(registro[col]) for col in columnas_valor):
            continue
        registros.append(registro)

    columnas_base = ["anio", "tabla", "pregunta", "ambito", "categoria"]
    return pd.DataFrame(registros)[columnas_base + columnas_valor]


## Funciones de transformación de Sabina: sector y CCAA

Adaptados para leer desde `pre_processed_2020_2024/EAL_15062026/{anio}` y para quedar dentro del selección automática común.


In [6]:
def process_sector_wide(anio, hoja):
    df = leer_csv_eal(anio, hoja)
    titulo = detectar_titulo(df, hoja)
    tabla, pregunta = extraer_tabla_pregunta(titulo)

    columnas_sector = ["TOTAL", "INDUSTRIA", "CONSTRUCCIÓN", "SERVICIOS"]
    patron_sector = "|".join(re.escape(x) for x in columnas_sector)

    mask_header = df.apply(
        lambda row: row.astype(str).str.contains(patron_sector, case=False, regex=True).sum() >= 2,
        axis=1,
    )
    header_row = df.index[mask_header][0]
    header = df.loc[header_row]

    value_cols = header[header.astype(str).str.contains(patron_sector, case=False, regex=True, na=False)].index.tolist()
    nombres_columnas = [normalizar_columna(x) for x in header[value_cols].tolist()]

    df_sector = df.loc[header_row + 1:, [df.columns[0]] + value_cols].copy()
    df_sector.columns = ["ambito"] + nombres_columnas
    df_sector = df_sector[df_sector["ambito"].notna()].copy()

    for col in nombres_columnas:
        df_sector[col] = convertir_numerico(df_sector[col])

    df_sector = df_sector.dropna(subset=["TOTAL"]).reset_index(drop=True)
    df_sector.insert(0, "anio", anio)
    df_sector.insert(1, "tabla", tabla)
    df_sector.insert(2, "pregunta", pregunta)
    return df_sector


def process_ccaa_long(anio, hoja):
    df = leer_csv_eal(anio, hoja)
    titulo = detectar_titulo(df, hoja)
    tabla, pregunta = extraer_tabla_pregunta(titulo)

    ccaa_keywords = [
        "TOTAL", "ANDALUCÍA", "ARAGÓN", "ASTURIAS (PRINCIPADO DE)", "BALEARS (ILLES)",
        "CANARIAS", "CANTABRIA", "CASTILLA-LA MANCHA", "CASTILLA Y LEÓN", "CATALUÑA",
        "COMUNITAT VALENCIANA", "EXTREMADURA", "GALICIA", "MADRID (COMUNIDAD DE)",
        "MURCIA (REGIÓN DE)", "NAVARRA (COMUNIDAD FORAL DE)", "PAÍS VASCO", "RIOJA, LA",
    ]
    patron_ccaa = "|".join(re.escape(x) for x in ccaa_keywords)

    mask_headers = df.apply(
        lambda row: row.astype(str).str.contains(patron_ccaa, case=False, regex=True).sum() >= 2,
        axis=1,
    )
    header_rows = df.index[mask_headers].tolist()
    bloques = []
    col_competencia = df.columns[0]

    for i, header_row in enumerate(header_rows):
        start_row = header_row + 1
        end_row = header_rows[i + 1] if i + 1 < len(header_rows) else len(df)
        header = df.loc[header_row]
        value_cols = [c for c in header[header.notna()].index.tolist() if c != col_competencia]
        nombres_ccaa = header[value_cols].tolist()

        block = df.loc[start_row:end_row - 1, [col_competencia] + value_cols].copy()
        block.columns = ["competencia"] + nombres_ccaa
        block = block[block["competencia"].notna()].copy()
        block_long = block.melt(
            id_vars="competencia",
            var_name="comunidad_autonoma",
            value_name="porcentaje",
        )
        bloques.append(block_long)

    df_final = pd.concat(bloques, ignore_index=True)
    df_final["porcentaje"] = convertir_numerico(df_final["porcentaje"])
    df_final = df_final.dropna(subset=["porcentaje"]).reset_index(drop=True)
    df_final.insert(0, "anio", anio)
    df_final.insert(1, "tabla", tabla)
    df_final.insert(2, "pregunta", pregunta)
    return df_final


## Selección automática por hoja

Esta es la pieza importante: según la hoja, se elige automáticamente la función adecuada.


In [7]:
CONFIG_HOJAS = {
    "EAL-16": {"tipo_transformacion": "eal16_auto"},
    "EAL-17": {"tipo_transformacion": "eal17_18"},
    "EAL-18": {"tipo_transformacion": "eal17_18"},
    "EAL-18a": {"tipo_transformacion": "simple", "fila_cabecera": 4, "fila_inicio_datos": 5},
    "EAL-18b": {"tipo_transformacion": "simple", "fila_cabecera": 5, "fila_inicio_datos": 7},
    "EAL-18c": {"tipo_transformacion": "simple", "fila_cabecera": 4, "fila_inicio_datos": 6},
    "EAL-19": {"tipo_transformacion": "simple", "fila_cabecera": 4, "fila_inicio_datos": 5},
    "EAL-20": {"tipo_transformacion": "simple", "fila_cabecera": 4, "fila_inicio_datos": 5},
    "EAL-21": {"tipo_transformacion": "sector"},
    "EAL-22": {"tipo_transformacion": "ccaa"},
    "EAL-23": {"tipo_transformacion": "grado"},
    "EAL-24": {"tipo_transformacion": "grado"},
    "EAL-25": {"tipo_transformacion": "grado"},
}


def transformar_hoja(anio, hoja):
    if hoja not in CONFIG_HOJAS:
        raise ValueError(f"No hay función configurada para la hoja {hoja}")

    config = CONFIG_HOJAS[hoja]
    tipo = config["tipo_transformacion"]

    if tipo == "eal16_auto":
        return process_eal16_auto(anio, hoja)
    if tipo == "grado":
        return process_grado_wide(anio, hoja)
    if tipo == "eal17_18":
        return process_eal17_18_wide(anio, hoja)
    if tipo == "simple":
        return process_hoja_ancha_simple(
            anio,
            hoja,
            fila_cabecera=config["fila_cabecera"],
            fila_inicio_datos=config["fila_inicio_datos"],
        )
    if tipo == "sector":
        return process_sector_wide(anio, hoja)
    if tipo == "ccaa":
        return process_ccaa_long(anio, hoja)

    raise ValueError(f"Tipo de transformación no reconocido: {tipo}")


def transformar_anios(anios, hojas):
    resultados = {}
    inventario = []

    for anio in anios:
        for hoja in hojas:
            if not existe_csv_pre_processed(anio, hoja):
                inventario.append({
                    "anio": anio,
                    "hoja": hoja,
                    "estado": "faltante",
                    "origen": "pre_processed",
                    "filas": 0,
                    "columnas": 0,
                    "detalle": "CSV no encontrado en pre_processed",
                })
                continue

            try:
                df = transformar_hoja(anio, hoja)
            except Exception as exc:
                inventario.append({
                    "anio": anio,
                    "hoja": hoja,
                    "estado": "error_transformacion",
                    "origen": "pre_processed",
                    "filas": 0,
                    "columnas": 0,
                    "detalle": f"{type(exc).__name__}: {exc}",
                })
                continue

            estado = "ok" if df.shape[0] > 0 else "sin_registros"
            resultados[(anio, hoja)] = df
            inventario.append({
                "anio": anio,
                "hoja": hoja,
                "estado": estado,
                "origen": "pre_processed",
                "filas": df.shape[0],
                "columnas": df.shape[1],
                "detalle": "",
            })

    return resultados, pd.DataFrame(inventario)


## Ejecución parametrizada

El notebook procesa los años detectados en `pre_processed`. Si una hoja no está extraída para un año concreto, queda marcada como `faltante` en el inventario.


In [8]:
resultados, inventario_transformacion = transformar_anios(ANIOS, HOJAS_FORMACION)

inventario_transformacion


,anio,hoja,estado,origen,filas,columnas,detalle
0,2023,EAL-16,ok,pre_processed,70,9,
1,2023,EAL-17,ok,pre_processed,32,11,
2,2023,EAL-18,ok,pre_processed,32,11,
3,2023,EAL-18a,ok,pre_processed,32,11,
4,2023,EAL-18b,ok,pre_processed,31,13,
5,2023,EAL-18c,ok,pre_processed,31,6,
6,2023,EAL-19,ok,pre_processed,6,9,
7,2023,EAL-20,ok,pre_processed,11,11,
8,2023,EAL-21,ok,pre_processed,11,8,
9,2023,EAL-22,ok,pre_processed,198,6,


## Ejecución parametrizada

El notebook procesa los años detectados en `pre_processed`. Si una hoja no está extraída para un año concreto, queda marcada como `faltante` en el inventario.


In [9]:
# Se crean variables df_2024_16, df_2024_17, etc. para evitar mezclar años.
# Además, df_16, df_17, etc. quedan como alias del último año procesado disponible.
for (anio, hoja), df in resultados.items():
    sufijo_hoja = hoja.replace("EAL-", "").replace("-", "_")
    globals()[f"df_{anio}_{sufijo_hoja}"] = df
    globals()[f"df_{sufijo_hoja}"] = df

print("Resumen de transformación por año y estado:")
display(inventario_transformacion.groupby(["anio", "estado", "origen"], dropna=False).size().reset_index(name="n_hojas"))

print("Dataframes del último año disponible:")
for nombre in sorted([k for k in globals() if re.match(r"^df_\d+[a-z]?$", k)]):
    print(nombre, globals()[nombre].shape)


Resumen de transformación por año y estado:


,anio,estado,origen,n_hojas
0,2023,ok,pre_processed,13


Dataframes del último año disponible:
df_16 (70, 9)
df_17 (32, 11)
df_18 (32, 11)
df_18a (32, 11)
df_18b (31, 13)
df_18c (31, 6)
df_19 (6, 9)
df_20 (11, 11)
df_21 (11, 8)
df_22 (198, 6)
df_23 (42, 9)
df_24 (70, 9)
df_25 (60, 9)


## Vista larga unificada

Esta vista permite juntar hojas con estructuras distintas en una base común de análisis. No sustituye a los dataframes específicos, pero sirve para explorar y filtrar por `anio`, `tabla`, `pregunta`, `ambito`, `categoria`, `variable` y `porcentaje`.


In [10]:
def normalizar_a_largo(df):
    df = df.copy()

    if "competencia" in df.columns and "comunidad_autonoma" in df.columns and "porcentaje" in df.columns:
        salida = df.rename(columns={
            "competencia": "ambito",
            "comunidad_autonoma": "variable",
        }).copy()
        salida["categoria"] = "comunidad_autonoma"
        return salida[["anio", "tabla", "pregunta", "ambito", "categoria", "variable", "porcentaje"]]

    id_vars = [col for col in ["anio", "tabla", "pregunta", "ambito", "categoria"] if col in df.columns]
    value_vars = [col for col in df.columns if col not in id_vars]
    largo = to_long_format(df, id_vars=id_vars, value_vars=value_vars, var_name="variable", value_name="porcentaje")

    if "categoria" not in largo.columns:
        largo["categoria"] = ""

    return largo[["anio", "tabla", "pregunta", "ambito", "categoria", "variable", "porcentaje"]]


df_unificado_largo = pd.concat(
    [normalizar_a_largo(df) for df in resultados.values()],
    ignore_index=True,
    sort=False,
)

print("df_unificado_largo:", df_unificado_largo.shape)
with pd.option_context("display.max_rows", 30, "display.max_columns", None, "display.max_colwidth", 100):
    display(df_unificado_largo.head(30))


df_unificado_largo: (2392, 7)


,anio,tabla,pregunta,ambito,categoria,variable,porcentaje
0,2023,EAL-16,"EMPRESAS, SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DI...",De dirección,,TOTAL,100.0
1,2023,EAL-16,"EMPRESAS, SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DI...",De trabajo en equipo,,TOTAL,100.0
2,2023,EAL-16,"EMPRESAS, SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DI...",De atención al público/ trato a clientes,,TOTAL,100.0
3,2023,EAL-16,"EMPRESAS, SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DI...",Administrativas de oficina,,TOTAL,100.0
4,2023,EAL-16,"EMPRESAS, SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DI...","De resolución de problemas (localización de problemas o fallos, análisis de sus causas y búsqued...",,TOTAL,100.0
5,2023,EAL-16,"EMPRESAS, SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DI...",En lenguas extranjeras,,TOTAL,100.0
6,2023,EAL-16,"EMPRESAS, SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DI...",Básicas de cálculo y/o comunicación oral o escrita,,TOTAL,100.0
7,2023,EAL-16,"EMPRESAS, SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DI...",Generales de tecnologías de la información,,TOTAL,100.0
8,2023,EAL-16,"EMPRESAS, SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DI...",Profesionales de tecnologías de la información,,TOTAL,100.0
9,2023,EAL-16,"EMPRESAS, SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DI...","Competencias técnicas, prácticas y otras específicas del puesto de trabajo",,TOTAL,100.0


## Guardado opcional

Por defecto no se guardan archivos. Cuando el equipo decida que este es el entregable común, cambiar `GUARDAR_OUTPUTS = True`.

Con el guardado activado se exportan:

- Un CSV por cada hoja transformada y año.
- Un inventario de transformación.
- Un CSV unificado en formato largo.


In [11]:
if GUARDAR_OUTPUTS:
    archivos_guardados = []

    for (anio, hoja), df in resultados.items():
        output_dir = PROCESSED_BASE / str(anio)
        output_dir.mkdir(parents=True, exist_ok=True)

        nombre_hoja = hoja.replace("-", "_").lower()
        output_path = output_dir / f"{nombre_hoja}_transformado.csv"
        df.to_csv(output_path, index=False, encoding="utf-8-sig")
        archivos_guardados.append(output_path)

    inventario_path = PROCESSED_BASE / "inventario_transformacion_eal.csv"
    largo_path = PROCESSED_BASE / "eal_formacion_unificado_largo.csv"

    PROCESSED_BASE.mkdir(parents=True, exist_ok=True)
    inventario_transformacion.to_csv(inventario_path, index=False, encoding="utf-8-sig")
    df_unificado_largo.to_csv(largo_path, index=False, encoding="utf-8-sig")

    archivos_guardados.extend([inventario_path, largo_path])

    print("Outputs guardados:")
    for ruta in archivos_guardados:
        print("-", ruta.relative_to(REPO_ROOT))
else:
    print("GUARDAR_OUTPUTS = False. No se han escrito archivos.")


GUARDAR_OUTPUTS = False. No se han escrito archivos.
